# ChEMBL Data Loading

This notebook is for figuring out how to load ChEMBL data. Official data loading scripts and stuff go in `ai/src`.

The work in here is largely based on investigative work done by Eman to understand the ChEMBL BACE-1 data.

In [ ]:
all_activities = []
limit = 1000
bace1_cols = [
    "molecule_chembl_id",
    "canonical_smiles",
    "pchembl_value",
    "bio_activity_class",
    "standard_value",
    "standard_units",
    "standard_type",
    "standard_relation",
    "potential_duplicate",
    "data_validity_comment",
    "assay_type",
]
base_url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"
params = {"target_chembl_id": "CHEMBL4822", "standard_type": "IC50", "limit": limit}

In [ ]:
import json
import urllib.request
from urllib.parse import urlencode

url = f"{base_url}?{urlencode(params)}"

while url:
    with urllib.request.urlopen(url) as response:
        data = json.load(response)

    records = data.get("activities", [])

    all_activities.extend(
        [{col: record.get(col) for col in bace1_cols} for record in records]
    )

    print(f"retrieved {len(all_activities)} records")

    next_path = data.get("page_meta", {}).get("next")
    url = f"https://www.ebi.ac.uk{next_path}" if next_path else None

print("\nTotal retrieved:", len(all_activities))

In [ ]:
import pandas as pd

df = pd.DataFrame.from_records(all_activities)
df.head()

In [ ]:
df["standard_relation"].value_counts(dropna=False)

In [ ]:
df["pchembl_value"] = pd.to_numeric(df["pchembl_value"], errors="coerce")
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

In [ ]:
mask = (
    (df["standard_relation"] == "=")
    & (df["standard_units"] == "nM")
    & (df["assay_type"] == "B")
    & (df["potential_duplicate"] == 0)
    & (df["data_validity_comment"].isna())
)

clean_df = df[mask].copy()
clean_df.dropna(subset=["canonical_smiles", "pchembl_value"], inplace=True)
clean_df.rename(columns={"pchembl_value": "pIC50"}, inplace=True)
clean_df["bio_activity_class"] = (clean_df["pIC50"] >= 6.0).astype(int)
clean_df.head()

In [ ]:
conflicting_mols = clean_df.groupby("molecule_chembl_id")[
    "bio_activity_class"
].nunique()
conflicting_mols = conflicting_mols[conflicting_mols > 1]

print(f"Number of molecules with conflicting classes: {len(conflicting_mols)}")

if len(conflicting_mols) > 0:
    display(
        clean_df[clean_df["molecule_chembl_id"].isin(conflicting_mols.index)]
        .sort_values("molecule_chembl_id")
        .head(10)
    )

In [ ]:
clean_df["pIC50"].describe()

In [ ]:
chembl_mol_df = clean_df.groupby("molecule_chembl_id", as_index=False).agg(
    {
        "canonical_smiles": "first",
        "pIC50": "median",
        # "bio_activity_class": "first"  # consider this instead of recalculating
    }
)
chembl_mol_df["bio_activity_class"] = (chembl_mol_df["pIC50"] >= 6.0).astype(int)
chembl_mol_df.head()

In [ ]:
chembl_mol_df["pIC50"].describe()

In [ ]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

chembl_mol_df["rdkit_mol"] = chembl_mol_df["canonical_smiles"].apply(Chem.MolFromSmiles)
chembl_mol_df["rdkit_canonical_smiles"] = chembl_mol_df["rdkit_mol"].apply(
    lambda mol: Chem.MolToSmiles(mol) if mol is not None else None
)
chembl_mol_df["scaffold"] = chembl_mol_df["rdkit_mol"].apply(
    lambda mol: MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
)

In [ ]:
display(chembl_mol_df)